# CRM Data Health Assessment
### Apache Spark · PySpark · Isolation Forest · K-Means

**Dataset:** IBM Telco Customer Churn  
**Objective:** Assess CRM data quality, identify unusual customer profiles, derive an anomaly-based Health Score, and explore customer segments.

**Before running:** Set `DATA_PATH` to the CSV file location in your environment. The private Databricks Volume path from the original notebook has been replaced with a configurable placeholder.

## 1. Environment and imports

Spark is used for data processing and K-Means. Isolation Forest is implemented with Scikit-learn, so the selected modelling data is converted to Pandas. This is appropriate for the 7,043-row dataset used here, but is not a distributed Isolation Forest implementation.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when, count, sum as spark_sum, isnan, round as spark_round, avg
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from sklearn.ensemble import IsolationForest
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
spark = SparkSession.builder.appName("CRM Data Health Assessment").getOrCreate()

# Replace with the location of the IBM Telco Customer Churn CSV.
DATA_PATH = "path/to/WA_Fn-UseC_-Telco-Customer-Churn.csv"

## 2. Load and inspect the dataset

In [ ]:
df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.show(5, truncate=False)
df.printSchema()

## 3. Data preparation and feature engineering

Trim whitespace, cast numeric fields, and create:
- `LifetimeRevenue` = `MonthlyCharges × tenure` (a derived proxy).
- `CustomerStatus` = `Loyal` for tenure of at least 24 months; otherwise `New`.

In [ ]:
for field in df.columns:
    df = df.withColumn(field, trim(col(field)))

df = (
    df.withColumn("SeniorCitizen", col("SeniorCitizen").cast("int"))
      .withColumn("tenure", col("tenure").cast("int"))
      .withColumn("MonthlyCharges", col("MonthlyCharges").cast("double"))
      .withColumn(
          "TotalCharges",
          when(col("TotalCharges") == "", None)
          .otherwise(col("TotalCharges").cast("double"))
      )
)

df = df.withColumn("LifetimeRevenue", spark_round(col("MonthlyCharges") * col("tenure"), 2))
df = df.withColumn("CustomerStatus", when(col("tenure") >= 24, "Loyal").otherwise("New"))

print("Rows:", df.count(), "| Columns:", len(df.columns))
df.select("customerID", "tenure", "MonthlyCharges", "TotalCharges", "LifetimeRevenue", "CustomerStatus").show(10)

## 4. Data quality assessment

### 4.1 Missing values

In [ ]:
missing_expressions = []
for field in df.schema.fields:
    name = field.name
    if field.dataType.simpleString() in ("int", "bigint", "float", "double"):
        missing_expressions.append(
            spark_sum(when(col(name).isNull() | isnan(col(name)), 1).otherwise(0)).alias(name)
        )
    else:
        missing_expressions.append(
            spark_sum(when(col(name).isNull(), 1).otherwise(0)).alias(name)
        )
df.select(missing_expressions).show(truncate=False)

### 4.2 Duplicate checks

In [ ]:
row_count = df.count()
unique_row_count = df.dropDuplicates().count()
duplicate_row_count = row_count - unique_row_count

duplicate_customer_ids = (
    df.groupBy("customerID").agg(count("*").alias("record_count"))
      .filter(col("record_count") > 1)
)

print("Total rows:", row_count)
print("Exact duplicate rows:", duplicate_row_count)
print("Customer IDs with multiple records:", duplicate_customer_ids.count())

### 4.3 Categorical and numerical validation

In [ ]:
categorical_columns = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod", "Churn", "CustomerStatus"
]
for name in categorical_columns:
    print(f"\\n{name}")
    df.select(name).distinct().show(truncate=False)

numeric_columns = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "LifetimeRevenue"]
for name in numeric_columns:
    print(f"\\n{name}")
    df.select(name).describe().show()

In [ ]:
# Empty outputs indicate no records violate these range checks.
df.filter(~col("SeniorCitizen").isin(0, 1)).show()
df.filter((col("tenure") < 0) | (col("tenure") > 72)).show()
df.filter(col("MonthlyCharges") < 0).show()
df.filter(col("TotalCharges") < 0).show()

## 5. Handle missing billing values

The project analysis found 11 missing `TotalCharges` values. They belonged to customers with `tenure = 0`, so the values were replaced with `0.0` to represent no accumulated billing.

In [ ]:
df_clean = df.withColumn(
    "TotalCharges",
    when(col("TotalCharges").isNull() & (col("tenure") == 0), 0.0)
    .otherwise(col("TotalCharges"))
)
print("Rows after cleaning:", df_clean.count())
print("Remaining missing TotalCharges:", df_clean.filter(col("TotalCharges").isNull()).count())

## 6. Prepare modelling features

In [ ]:
features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "LifetimeRevenue"]
df_model = df_clean.select("customerID", *features)
df_model.show(5)

## 7. Anomaly detection — Isolation Forest

Configuration: 100 estimators, `contamination="auto"`, and `random_state=42`. Higher `decision_function` values indicate more typical observations. The rescaled score therefore assigns higher Health Scores to more typical profiles.

**Scalability note:** `toPandas()` collects the selected records on the driver. Do not use this unchanged for datasets too large to fit in driver memory.

In [ ]:
model_input = df_model.toPandas()
X = model_input[features].copy()

isolation_forest = IsolationForest(n_estimators=100, contamination="auto", random_state=42)
isolation_forest.fit(X)

scored_customers = pd.DataFrame({
    "customerID": model_input["customerID"],
    "Anomaly": isolation_forest.predict(X),
    "Score": isolation_forest.decision_function(X)
})

score_range = scored_customers["Score"].max() - scored_customers["Score"].min()
if score_range == 0:
    scored_customers["HealthScore"] = 100.0
else:
    scored_customers["HealthScore"] = (
        (scored_customers["Score"] - scored_customers["Score"].min()) / score_range
    ) * 100

scored_customers.head()

### 7.1 Results and Health Score interpretation

The Health Score is an anomaly-based score from 0–100. It is **not** a full CRM data-quality score, customer satisfaction score, churn probability, or measure of loyalty.

In [ ]:
normal_count = int((scored_customers["Anomaly"] == 1).sum())
anomaly_count = int((scored_customers["Anomaly"] == -1).sum())
anomaly_rate = anomaly_count / len(scored_customers) * 100
average_health_score = scored_customers["HealthScore"].mean()

print("Total customers:", len(scored_customers))
print("Normal profiles:", normal_count)
print("Unusual profiles:", anomaly_count)
print(f"Anomaly rate: {anomaly_rate:.2f}%")
print(f"Average Health Score: {average_health_score:.2f}")

In [ ]:
counts = scored_customers["Anomaly"].value_counts().reindex([1, -1], fill_value=0)
plt.figure(figsize=(7, 4))
plt.bar(["Normal", "Unusual"], counts.values)
plt.title("Isolation Forest: Customer Profile Classification")
plt.xlabel("Profile classification")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

In [ ]:
scored_spark = spark.createDataFrame(scored_customers)
result_df = df_clean.join(scored_spark, on="customerID", how="inner")
result_df.createOrReplaceTempView("crm_health")
result_df.select("customerID", "Anomaly", "Score", "HealthScore", "CustomerStatus").show(10, truncate=False)

### 7.2 Spark SQL exploration

In [ ]:
spark.sql("SELECT CustomerStatus, COUNT(*) AS customers FROM crm_health GROUP BY CustomerStatus ORDER BY CustomerStatus").show()

In [ ]:
spark.sql("SELECT customerID, Anomaly, HealthScore, CustomerStatus FROM crm_health ORDER BY HealthScore ASC LIMIT 10").show(truncate=False)

## 8. Customer segmentation — K-Means with Spark MLlib

K-Means is configured with `k=3`. This preserves the original project's unscaled-feature approach. **Important limitation:** the input variables have very different ranges, so charge variables can dominate Euclidean distance. Treat these clusters and the silhouette score as preliminary; standardise the features and compare results before making strong claims.

In [ ]:
assembler = VectorAssembler(inputCols=features, outputCol="features", handleInvalid="skip")
kmeans_input = assembler.transform(df_clean)

kmeans = KMeans(k=3, seed=42, featuresCol="features", predictionCol="prediction")
kmeans_model = kmeans.fit(kmeans_input)
result_kmeans = kmeans_model.transform(kmeans_input)

result_kmeans.select("customerID", "prediction").show(10)

In [ ]:
cluster_summary = (
    result_kmeans.groupBy("prediction")
    .agg(
        count("*").alias("Customers"),
        avg("tenure").alias("Avg_Tenure"),
        avg("MonthlyCharges").alias("Avg_MonthlyCharges"),
        avg("TotalCharges").alias("Avg_TotalCharges"),
        avg("LifetimeRevenue").alias("Avg_LifetimeRevenue")
    )
    .orderBy("prediction")
)
cluster_summary.show(truncate=False)

In [ ]:
cluster_counts = result_kmeans.groupBy("prediction").count().orderBy("prediction").toPandas()
plt.figure(figsize=(7, 4))
plt.bar(cluster_counts["prediction"].astype(str), cluster_counts["count"])
plt.title("K-Means Customer Segments")
plt.xlabel("Cluster")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

## 9. Model evaluation

The Isolation Forest anomaly rate is descriptive, not an accuracy metric, because ground-truth anomaly labels are not available. K-Means is evaluated using silhouette score and WCSS; interpret the silhouette score cautiously because the features are currently unscaled.

In [ ]:
evaluator = ClusteringEvaluator(
    predictionCol="prediction", featuresCol="features", metricName="silhouette"
)
silhouette_score = evaluator.evaluate(result_kmeans)
wcss = kmeans_model.summary.trainingCost

print("ISOLATION FOREST")
print("Normal profiles:", normal_count)
print("Unusual profiles:", anomaly_count)
print(f"Anomaly rate: {anomaly_rate:.2f}%")
print(f"Average Health Score: {average_health_score:.2f}")

print("\\nK-MEANS")
print(f"Silhouette Score (unscaled features): {silhouette_score:.4f}")
print(f"WCSS: {wcss:,.2f}")

## 10. Conclusion and future improvements

This project demonstrates a CRM analytics workflow combining data quality checks, anomaly detection, customer scoring, and customer segmentation.

**Key next steps**
1. Standardise numerical features before K-Means and compare cluster quality.
2. Investigate unusual profiles using business context; unusual does not automatically mean incorrect.
3. Validate anomaly detection with labelled examples or controlled anomaly injection.
4. Consider distributed anomaly-detection methods for substantially larger datasets.
5. Run all cells from top to bottom after setting `DATA_PATH`; use freshly generated outputs rather than relying on old saved results.

**Reproducibility:** Metrics may vary with library versions and execution environments. Treat the outputs generated by running this notebook as the current results.